In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

In [2]:

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

d:\_SELF_MASTERs\_UDACITY_MASTERS\env_python\pytorch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [157]:
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.to(device)

print("Model parameters (total):", sum(p.numel() for p in model.parameters()))

Model parameters (total): 134515008


In [158]:
ALL_WORDS = [
    "idea", "glow", "rust", "maze", "echo", "wisp", "veto", "lush", "gaze", "knit", "fume", "plow",
    "void", "oath", "grim", "crisp", "lunar", "fable", "quest", "verge", "brawn", "elude", "aisle",
    "ember", "crave", "ivory", "mirth", "knack", "wryly", "onset", "mosaic", "velvet", "sphinx",
    "radius", "summit", "banner", "cipher", "glisten", "mantle", "scarab", "expose", "fathom",
    "tavern", "fusion", "relish", "lantern", "enchant", "torrent", "capture", "orchard", "eclipse",
    "frescos", "triumph", "absolve", "gossipy", "prelude", "whistle", "resolve", "zealous",
    "mirage", "aperture", "sapphire",
]

In [164]:
def generate_records():
    for word in ALL_WORDS:
        yield {
            "prompt": (
                f"You spell words with hyphens between the letters like this W-O-R-D.\nWord:\n{word}\n\n"
                + "Spelling:\n"
            ),
            "completion": "-".join(word).upper() + ".",  # Of the form W-O-R-D.
        }


ds = Dataset.from_generator(generate_records)
ds[0]

{'prompt': 'You spell words with hyphens between the letters like this W-O-R-D.\nWord:\nidea\n\nSpelling:\n',
 'completion': 'I-D-E-A.'}

In [165]:
ds = ds.train_test_split(test_size=0.25, seed=42)

In [120]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 46
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 16
    })
})

In [121]:
train_ds = ds["train"]
test_ds = ds["test"]

print(len(train_ds))
print(len(test_ds))

46
16


# base 

In [122]:
def check_spelling(
    model, tokenizer, prompt: str, actual_spelling: str, max_new_tokens: int = 20
):
    inputs = tokenizer(prompt,return_tensors="pt").to(device)
    gen_out = model.generate(**inputs,max_new_tokens=max_new_tokens,use_cache=False)
    decoded_out = tokenizer.decode(gen_out[0],skip_special_tokens=True)
    proposed_spelling = decoded_out.split("Spelling:")[-1].strip().split("\n")[0].strip()
    #print(proposed_spelling)
    #print(f"Proposed: {proposed_spelling} | Actual: {actual_spelling} ")
    actual_spelling = actual_spelling.strip()
    #_proposed_spelling = proposed_spelling.replace("-", "")
    #_actual_spelling = actual_spelling.replace("-", "")
    num_correct = sum(1 for a, b in zip(actual_spelling, proposed_spelling) if a == b)


    print(
        f"Proposed: {proposed_spelling} | Actual: {actual_spelling} "
        f"| Matches: {'✅' if proposed_spelling == actual_spelling else '❌'}"
    )

    return num_correct / len(actual_spelling)


In [123]:
print(ds["test"][0]["completion"])
check_spelling(
    model=model,
    tokenizer=tokenizer,
    prompt=ds["test"][0]["prompt"],
    actual_spelling=ds["test"][0]["completion"],
)

W-R-Y-L-Y.
Proposed: wry | Actual: W-R-Y-L-Y. | Matches: ❌


0.0

In [100]:
correct = 0.0
for example in ds["train"]:
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    correct += result

print(f"{correct}/{len(ds['train'])} words correct")

Proposed: sphinx | Actual: S-P-H-I-N-X. | Matches: ❌
Proposed: brawn | Actual: B-R-A-W-N. | Matches: ❌
Proposed: goss | Actual: G-O-S-S-I-P-Y. | Matches: ❌
Proposed: enchant | Actual: E-N-C-H-A-N-T. | Matches: ❌
Proposed: tavern | Actual: T-A-V-E-R-N. | Matches: ❌
Proposed: whistle | Actual: W-H-I-S-T-L-E. | Matches: ❌
Proposed: W-O-R-D | Actual: C-A-P-T-U-R-E. | Matches: ❌
Proposed: echo | Actual: E-C-H-O. | Matches: ❌
Proposed: mirth | Actual: M-I-R-T-H. | Matches: ❌
Proposed: cris | Actual: C-R-I-S-P. | Matches: ❌
Proposed: zeal | Actual: Z-E-A-L-O-U-S. | Matches: ❌
Proposed:  | Actual: E-M-B-E-R. | Matches: ❌
Proposed: scarab | Actual: S-C-A-R-A-B. | Matches: ❌
Proposed:  | Actual: K-N-I-T. | Matches: ❌
Proposed: resolve | Actual: R-E-S-O-L-V-E. | Matches: ❌
Proposed: velvet | Actual: V-E-L-V-E-T. | Matches: ❌
Proposed:  | Actual: A-B-S-O-L-V-E. | Matches: ❌
Proposed: lunar | Actual: L-U-N-A-R. | Matches: ❌
Proposed: maze | Actual: M-A-Z-E. | Matches: ❌
Proposed:  | Actual: S-U-M-M

# Lets try SFT with LORA

In [124]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params BEFORE: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)


Trainable params BEFORE: 134,515,008 / 134,515,008 (100.00%)


In [166]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=8,
    lora_dropout=0.4,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

d:\_SELF_MASTERs\_UDACITY_MASTERS\env_python\pytorch_env\Lib\site-packages\peft\mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
d:\_SELF_MASTERs\_UDACITY_MASTERS\env_python\pytorch_env\Lib\site-packages\peft\tuners\tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [167]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params AFTER: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)

Trainable params AFTER: 1,843,200 / 136,358,208 (1.35%)


In [168]:
?SFTConfig

Init signature:
SFTConfig(
    output_dir: Optional[str] = None,
    overwrite_output_dir: bool = False,
    do_train: bool = False,
    do_eval: bool = False,
    do_predict: bool = False,
    eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no',
    prediction_loss_only: bool = False,
    per_device_train_batch_size: int = 8,
    per_device_eval_batch_size: int = 8,
    per_gpu_train_batch_size: Optional[int] = None,
    per_gpu_eval_batch_size: Optional[int] = None,
    gradient_accumulation_steps: int = 1,
    eval_accumulation_steps: Optional[int] = None,
    eval_delay: Optional[float] = 0,
    torch_empty_cache_steps: Optional[int] = None,
    learning_rate: float = 2e-05,
    weight_decay: float = 0.0,
    adam_beta1: float = 0.9,
    adam_beta2: float = 0.999,
    adam_epsilon: float = 1e-08,
    max_grad_norm: float = 1.0,
    num_train_epochs: float = 3.0,
    max_steps: int = -1,
    lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType,

In [169]:
from transformers.trainer_utils import SchedulerType
print(SchedulerType._member_names_)

['LINEAR', 'COSINE', 'COSINE_WITH_RESTARTS', 'POLYNOMIAL', 'CONSTANT', 'CONSTANT_WITH_WARMUP', 'INVERSE_SQRT', 'REDUCE_ON_PLATEAU', 'COSINE_WITH_MIN_LR', 'WARMUP_STABLE_DECAY']


In [170]:
output_dir = "tensorboard"
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=50,
    learning_rate=5e-5,
    logging_steps=8,
    eval_strategy="steps",
    eval_steps=8,
    save_strategy="no",
    report_to=[],
    fp16=False,
    lr_scheduler_type="inverse_sqrt",
)

In [171]:
trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=training_args,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [176]:
trainer.train()

Step,Training Loss,Validation Loss
8,0.481100,0.632269
16,0.453700,0.628582
24,0.456400,0.625496
32,0.449500,0.623055
40,0.424000,0.620271
48,0.434900,0.617465
56,0.424600,0.615198
64,0.393400,0.612864
72,0.408500,0.610631
80,0.400300,0.609247


TrainOutput(global_step=150, training_loss=0.39088642040888466, metrics={'train_runtime': 67.2531, 'train_samples_per_second': 34.199, 'train_steps_per_second': 2.23, 'total_flos': 69766590152448.0, 'train_loss': 0.39088642040888466})

In [177]:
proportion_correct = 0.0

for example in ds["train"]:
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    proportion_correct += result
num = len(ds["train"])
print(f"{proportion_correct}/{num} words correct")

Proposed: S-P-H-I-N-X. | Actual: S-P-H-I-N-X. | Matches: ✅
Proposed: B-R-A-W-N. | Actual: B-R-A-W-N. | Matches: ✅
Proposed: G-O-S-H-O-P-I-Y. | Actual: G-O-S-S-I-P-Y. | Matches: ❌
Proposed: E-N-C-H-A-N-T. | Actual: E-N-C-H-A-N-T. | Matches: ✅
Proposed: T-A-A-N-B-R. | Actual: T-A-V-E-R-N. | Matches: ❌
Proposed: W-H-I-S-T-E. | Actual: W-H-I-S-T-L-E. | Matches: ❌
Proposed: C-U-P-A-R-E. | Actual: C-A-P-T-U-R-E. | Matches: ❌
Proposed: E-C-H-O-R-D. | Actual: E-C-H-O. | Matches: ❌
Proposed: M-I-R-T-H. | Actual: M-I-R-T-H. | Matches: ✅
Proposed: C-R-I-S-P. | Actual: C-R-I-S-P. | Matches: ✅
Proposed: Z-E-A-L-O-U-S. | Actual: Z-E-A-L-O-U-S. | Matches: ✅
Proposed: E-M-B-U-R-E. | Actual: E-M-B-E-R. | Matches: ❌
Proposed: S-C-A-R-B-E. | Actual: S-C-A-R-A-B. | Matches: ❌
Proposed: W-I-N-K. | Actual: K-N-I-T. | Matches: ❌
Proposed: R-E-S-I-L-O-S. | Actual: R-E-S-O-L-V-E. | Matches: ❌
Proposed: V-E-L-V-E-T. | Actual: V-E-L-V-E-T. | Matches: ✅
Proposed: A-B-E-U-R-V-E. | Actual: A-B-S-O-L-V-E. | Matches:

In [178]:
proportion_correct = 0.0
num_examples = len(ds["test"])

for example in ds["test"]:
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    proportion_correct += result

print(f"{proportion_correct}/{num_examples}.0 words correct")

Proposed: W-O-R-Y-L-Y. | Actual: W-R-Y-L-Y. | Matches: ❌
Proposed: G-L-I-N-E-S. | Actual: G-L-I-S-T-E-N. | Matches: ❌
Proposed: C-A-S-Q-E-L. | Actual: Q-U-E-S-T. | Matches: ❌
Proposed: C-E-R-A-V-E. | Actual: C-R-A-V-E. | Matches: ❌
Proposed: L-U-S-I-S-H. | Actual: L-U-S-H. | Matches: ❌
Proposed: F-A-L-I-C-E. | Actual: F-A-B-L-E. | Matches: ❌
Proposed: K-N-A-R-K-E. | Actual: K-N-A-C-K. | Matches: ❌
Proposed: T-I-R-U-M-P-H-I-L-E. | Actual: T-R-I-U-M-P-H. | Matches: ❌
Proposed: S-A-P-I-C-R-H. | Actual: S-A-P-P-H-I-R-E. | Matches: ❌
Proposed: E-X-P-S-E-R. | Actual: E-X-P-O-S-E. | Matches: ❌
Proposed: F-S-R-C-O-S-S. | Actual: F-R-E-S-C-O-S. | Matches: ❌
Proposed: W-I-P-S. | Actual: W-I-S-P. | Matches: ❌
Proposed: M-I-R-A-G-E. | Actual: M-I-R-A-G-E. | Matches: ✅
Proposed: I-V-O-R-Y. | Actual: I-V-O-R-Y. | Matches: ✅
Proposed: O-N-S-H-E-R-D. | Actual: O-N-S-E-T. | Matches: ❌
Proposed: E-L-E-U-D. | Actual: E-L-U-D-E. | Matches: ❌
11.0125/16.0 words correct


# So only 2 out of all words were correct

I will revisit this once I learn a bit more , i think i will checkout HF's transformer course 